# STRATA com base fina T=1 e três níveis: Llama-3.1-8B contra k-quants com imatrix

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução* → **A100** com **RAM alta** (ou L4). Depois *Executar tudo*.

Na rodada anterior (seção 25 dos resultados), a base fina pôs o STRATA no nível dos k-quants com imatrix nos níveis 1 e 2 do 8B, mas cada versão foi boa num nível só: T=1 no nível 1 (KL 7,59 contra 8,60 do Q5_K_M), T=2 no nível 2 (3,16 contra 3,47 do Q6_K). Um arquivo só pode ter uma base. No Llama-3.2-1B (container), mudar os pesos da T=2 para favorecer o nível 1 não ajudou (KL 14,4 contra 14,3), então a candidata a versão única é a **T=1 com um nível 2** que divide ao meio a célula da grade fina (novo `--K 2`): no 1B, nível 1 12,45 (Q5_K_M 13,9) e nível 2 6,11 (Q6_K 7,3; a T=2 dava 5,4). Este notebook mede esse nível 2 no 8B, e põe o **Q8_0** na tabela, para comparar com uma curva medida dos k-quants em vez de extrapolada.

O notebook:
1. compila o llama.cpp (commit fixo) com CUDA e os tipos do STRATA;
2. gera BF16, a imatrix (wikitext-2 de *treino*) e os k-quants com imatrix: Q4_K_M, Q5_K_M, Q6_K e Q8_0;
3. gera o STRATA com base fina T=1, níveis 0, 1 e 2;
4. mede perplexidade no wikitext-2 de teste inteiro e KL contra o BF16, na GPU.

Ocupa ~65 GB de disco no pico.

In [ ]:
CHUNKS = 0               # trechos de 512 tokens na perplexidade (0 = o texto inteiro)
KLD_CHUNKS = 100         # trechos na divergência KL (~65 MB de disco por trecho)
IMATRIX_CHUNKS = 125     # trechos de 512 tokens do wikitext de treino para a imatrix
MODELO_HF = "unsloth/Meta-Llama-3.1-8B"

In [ ]:
import os, re, json, shutil, subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap,driver_version', '--format=csv,noheader'], capture_output=True, text=True)
assert gpu.returncode == 0, 'Sem GPU NVIDIA: troque o tipo de ambiente de execução para A100 (ou L4).'
GPU = gpu.stdout.strip().splitlines()[0]
N = os.cpu_count()
RAM = int(next(l for l in open('/proc/meminfo') if l.startswith('MemTotal')).split()[1]) / 1e6
DISCO = shutil.disk_usage('/content').free / 1e9
print(f'GPU: {GPU} | CPU: {N} threads | RAM {RAM:.0f} GB | disco livre {DISCO:.0f} GB')
if DISCO < 85:
    print('Aviso: pouco disco. Ponha KLD_CHUNKS menor.')

In [ ]:
BRANCH = "claude/project-thread-qjksmf"
LLAMA_SHA = "2a667032dc788c3a071d538e0b572e45a3f12731"   # o commit em que o patch dos tipos foi feito
!rm -rf strata && git clone -q --depth 1 -b $BRANCH https://github.com/Cantyll/strata.git
!rm -rf llama.cpp && git init -q llama.cpp && cd llama.cpp && git remote add origin https://github.com/ggml-org/llama.cpp.git && git fetch -q --depth 1 origin $LLAMA_SHA && git checkout -q FETCH_HEAD
!sh strata/kernel/ggml/aplicar.sh llama.cpp
!pip install -q -U transformers safetensors
!pip install -q -e llama.cpp/gguf-py
# o fim do log da compilação fica visível, para aparecer qualquer erro
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DGGML_NATIVE=ON 2>&1 | tail -3 && cmake --build build -j $(nproc) --target llama-quantize llama-perplexity llama-imatrix 2>&1 | tail -15
!ls -la llama.cpp/build/bin | grep -E 'llama-(perplexity|quantize|imatrix)$'

In [ ]:
import urllib.request, zipfile, io
try:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen('https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip').read()))
    open('/content/wiki.test.raw', 'wb').write(z.read('wikitext-2-raw/wiki.test.raw'))
    open('/content/wiki.train.raw', 'wb').write(z.read('wikitext-2-raw/wiki.train.raw'))
    TEXTO = 'wikitext-2 raw'
except Exception as e:
    print('download do wikitext raw falhou:', e)
    base = 'https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/'
    urllib.request.urlretrieve(base + 'test.txt', '/content/wiki.test.raw')
    urllib.request.urlretrieve(base + 'train.txt', '/content/wiki.train.raw')
    TEXTO = 'wikitext-2 (não-raw, cópia do PyTorch)'
print(TEXTO, os.path.getsize('/content/wiki.test.raw'), 'bytes de teste,', os.path.getsize('/content/wiki.train.raw'), 'de treino')

Os GGUF. A base fina lê os pesos do próprio GGUF BF16 (já na ordem do llama.cpp), então os pesos originais são apagados depois da conversão.

In [ ]:
from huggingface_hub import snapshot_download
src = snapshot_download(MODELO_HF, allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
G = '/content/gguf'
os.makedirs(G, exist_ok=True)
!python llama.cpp/convert_hf_to_gguf.py {src} --outtype bf16 --outfile {G}/BF16.gguf 2>&1 | tail -1
!rm -rf {src}
!llama.cpp/build/bin/llama-imatrix -m {G}/BF16.gguf -f /content/wiki.train.raw -ngl 99 -c 512 --chunks {IMATRIX_CHUNKS} -o {G}/imatrix.gguf > {G}/imatrix.log 2>&1
!tail -2 {G}/imatrix.log
!llama.cpp/build/bin/llama-quantize {G}/BF16.gguf {G}/Q8_0.gguf Q8_0 > {G}/Q8_0.log 2>&1
for q in ['Q4_K_M', 'Q5_K_M', 'Q6_K']:
    !llama.cpp/build/bin/llama-quantize --imatrix {G}/imatrix.gguf {G}/BF16.gguf {G}/{q}_imat.gguf {q} > {G}/{q}_imat.log 2>&1
%cd /content/strata
!python tests/kernel/strata_base_fina.py --bf16 {G}/BF16.gguf --gguf {G}/Q4_K_M_imat.gguf --imatrix {G}/imatrix.gguf --T 1 --K 2 --out {G}/FINA1_N0.gguf {G}/FINA1_N1.gguf {G}/FINA1_N2.gguf 2>&1 | grep -v Warning | tail -3
%cd /content
!ls -la {G}/*.gguf
!df -h /content | tail -1

In [ ]:
def ppl_de(m, chunks=0):
    c = f'--chunks {chunks}' if chunks else ''
    out = !llama.cpp/build/bin/llama-perplexity -m {G}/{m}.gguf -f /content/wiki.test.raw -c 512 -ngl 99 {c} 2>&1
    r = re.search(r'Final estimate: PPL = ([0-9.]+)\s*\+/- ([0-9.]+)', '\n'.join(out))
    if not r:
        print(f'Erro ao obter a PPL de {m}. Fim da saída:')
        print('\n'.join(out[-10:]))
        return None
    return (float(r.group(1)), float(r.group(2)))

Perplexidade no wikitext-2 de teste (contexto 512) e divergência KL contra o BF16 nos primeiros `KLD_CHUNKS` trechos.

In [ ]:
MODELOS = ['BF16', 'Q4_K_M_imat', 'FINA1_N0', 'Q5_K_M_imat', 'FINA1_N1', 'Q6_K_imat', 'FINA1_N2', 'Q8_0']
ppl, kld = {}, {}
for m in MODELOS:
    ppl[m] = ppl_de(m, CHUNKS)
    if ppl[m]:
        print(m, ppl[m], f'ΔPPL {100 * (ppl[m][0] / ppl["BF16"][0] - 1):+.2f}%')
!llama.cpp/build/bin/llama-perplexity -m {G}/BF16.gguf -f /content/wiki.test.raw -c 512 -ngl 99 --chunks {KLD_CHUNKS} --kl-divergence-base /content/kld_base.bin > /dev/null 2>&1
for m in MODELOS[1:]:
    out = !llama.cpp/build/bin/llama-perplexity -m {G}/{m}.gguf --kl-divergence-base /content/kld_base.bin --kl-divergence -ngl 99 2>&1
    txt = '\n'.join(out)
    k_ = re.search(r'Mean +KLD: +([0-9.]+) ± +([0-9.]+)', txt)
    top = re.search(r'Same top p: +([0-9.]+) ± +([0-9.]+) %', txt)
    if not k_:
        print(f'Erro na KL de {m}:', '\n'.join(out[-8:]))
        continue
    kld[m] = (float(k_.group(1)), float(k_.group(2)), float(top.group(1)))
    print(m, 'KL ×1000', round(1000 * kld[m][0], 2), 'mesmo 1º token', kld[m][2], '%')
!rm -f /content/kld_base.bin

In [ ]:
from IPython.display import Markdown, display
tam = {m: os.path.getsize(f'{G}/{m}.gguf') / 2**30 for m in MODELOS}
L = [f'# STRATA com base fina T=1, três níveis (Llama-3.1-8B, llama.cpp CUDA)', '',
     f'GPU: {GPU}. {TEXTO} de teste, contexto 512{f", {CHUNKS} trechos" if CHUNKS else ", inteiro"}; KL contra o BF16 nos {KLD_CHUNKS} primeiros trechos; '
     f'imatrix com {IMATRIX_CHUNKS} trechos do texto de treino. FINA1 = base fina T=1, nível 2 por divisão da célula fina. Q8_0 sem imatrix (o tipo não usa).', '',
     '| modelo | GGUF (GiB) | PPL | ΔPPL | KL (×1000) | mesmo 1º token |', '|---|---:|---:|---:|---:|---:|']
for m in MODELOS:
    p, e = ppl.get(m) or (float('nan'), float('nan'))
    kl = f'{1000 * kld[m][0]:.2f} ± {1000 * kld[m][1]:.2f} | {kld[m][2]:.2f}%' if m in kld else ' | '
    dp = f'{100 * (p / ppl["BF16"][0] - 1):+.2f}%' if ppl.get('BF16') else ''
    L.append(f'| {m} | {tam[m]:.3f} | {p:.4f} ± {e:.4f} | {dp} | {kl} |')
L += ['', 'STRATA contra o k-quant com imatrix do mesmo nível (abaixo de 1 = mais perto do original):', '',
      '| comparação | tamanho | KL |', '|---|---:|---:|']
for a, b in [('FINA1_N0', 'Q4_K_M_imat'), ('FINA1_N1', 'Q5_K_M_imat'), ('FINA1_N2', 'Q6_K_imat')]:
    kl = f'{kld[a][0] / kld[b][0]:.2f}' if a in kld and b in kld else ''
    L.append(f'| {a} / {b} | {tam[a] / tam[b]:.3f} | {kl} |')
L += ['', 'Rodada anterior (seção 25, A100): Q4_K_M 26,26, Q5_K_M 8,60, Q6_K 3,47; FINA1 T=1: N0 33,11, N1 7,59; FINA2 T=2 pesos 1 2 4: N0 34,24, N1 8,69, N2 3,16 (KL ×1000).']
os.makedirs('/content/res', exist_ok=True)
open('/content/res/strata_8b_fina2.md', 'w').write('\n'.join(L) + '\n')
json.dump({'texto': TEXTO, 'gpu': GPU, 'chunks': CHUNKS, 'kld_chunks': KLD_CHUNKS, 'imatrix_chunks': IMATRIX_CHUNKS, 'ppl': ppl, 'kld': kld,
           'gguf_gib': tam}, open('/content/res/strata_8b_fina2.json', 'w'), indent=1)
display(Markdown('\n'.join(L)))

Cole a tabela na thread (ou baixe os arquivos abaixo).

In [ ]:
from google.colab import files
files.download('/content/res/strata_8b_fina2.md')
files.download('/content/res/strata_8b_fina2.json')